In [33]:
# 0

import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# 1

H = dict(code=7, name=8, item_code=9, item_name=13)
DATA_START = 14
N_ITEM = 19

def load_sheet(path, sheet='Financials'):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    codes  = raw.iloc[H['code'],      1:].to_numpy()
    cnames = raw.iloc[H['name'],      1:].to_numpy()
    icodes = raw.iloc[H['item_code'], 1:].to_numpy()
    inames = raw.iloc[H['item_name'], 1:].to_numpy()
    dates  = raw.iloc[DATA_START:, 0].to_numpy()
    vals   = raw.iloc[DATA_START:, 1:].apply(pd.to_numeric, errors='coerce').to_numpy()
    return codes, cnames, icodes, inames, dates, vals

def to_tidy(codes, cnames, icodes, inames, dates, vals, n_item=N_ITEM):
    n_col, n_date = len(codes), len(dates)
    assert n_col % n_item == 0, f'{n_col} not divisible by {n_item}'
    n_comp = n_col // n_item
    icode_blk = icodes.reshape(n_comp, n_item)
    assert (icode_blk == icode_blk[0]).all(), 'item order differs across companies'
    comp_code = codes.reshape(n_comp, n_item)[:, 0]
    comp_name = cnames.reshape(n_comp, n_item)[:, 0]
    item_code = icode_blk[0]
    item_name = inames.reshape(n_comp, n_item)[0]
    cube = vals.reshape(n_date, n_comp, n_item)
    flat = cube.reshape(n_date * n_comp, n_item)
    out = pd.DataFrame(flat, columns=list(item_code))
    out.insert(0, 'date', np.repeat(dates, n_comp))
    out.insert(0, 'name', np.tile(comp_name, n_date))
    out.insert(0, 'code', np.tile(comp_code, n_date))
    out['date'] = pd.to_datetime(out['date'], format='%Y-%m') + pd.offsets.MonthEnd(0)
    out = out.sort_values(['code', 'date']).reset_index(drop=True)
    item_map = pd.DataFrame({'item_code': item_code, 'item_name': item_name})
    return out, item_map

def excel_to_parquet(path, sheet='Financials', out_path=None, map_path='item_map.csv'):
    path = Path(path)
    out_path = Path(out_path) if out_path else path.with_suffix('.parquet')
    if out_path.exists():
        print(f'{out_path.name} exists, skip')
        return pd.read_parquet(out_path)
    tidy, item_map = to_tidy(*load_sheet(path, sheet))
    tidy.to_parquet(out_path)
    if not Path(map_path).exists():
        item_map.to_csv(map_path, index=False, encoding='utf-8-sig')
    print(f'{path.name} -> {out_path.name}  shape {tidy.shape}  '
          f'n_comp {tidy["code"].nunique()}  n_date {tidy["date"].nunique()}')
    return tidy
k200data = excel_to_parquet('k200data.xlsx', sheet='Financials', out_path='k200data.parquet')

k200data.parquet exists, skip


In [35]:
# 2

RENAME = {
    'M121000.M': 'saleq',
    'M111100.M': 'actq',
    'M111400.M': 'rectq',
    'M111500.M': 'invtq',
    'M111800.M': 'ppentq',
    'M111000.M': 'atq',
    'M113100.M': 'lctq',
    'M113300.M': 'dlcq',
    'M113200.M': 'apq',
    'M113700.M': 'dlttq',
    'M113000.M': 'ltq',
    'M115000.M': 'seqq',
    'M121100.M': 'cogsq',
    'M121300.M': 'xsgaq',
}

FVOL_COLS = ['actq','rectq','invtq','ppentq','atq','lctq','dlcq',
             'apq','dlttq','ltq','seqq','xoprq','cogsq','xsgaq']

comp_raw = pd.read_parquet('k200data.parquet')
comp_raw = comp_raw.rename(columns=RENAME)
comp_raw['xoprq'] = comp_raw['cogsq'] + comp_raw['xsgaq']
comp_raw = comp_raw[['code','date','saleq'] + FVOL_COLS]
comp_raw = comp_raw.sort_values(['code','date']).reset_index(drop=True)

print('comp_raw:', comp_raw.shape)
print('n_comp:', comp_raw['code'].nunique(), ' n_date:', comp_raw['date'].nunique())
print((comp_raw[['saleq'] + FVOL_COLS].notna().mean().round(3)).to_dict())

comp_raw: (26166, 17)
n_comp: 178  n_date: 147
{'saleq': 0.55, 'actq': 0.56, 'rectq': 0.56, 'invtq': 0.537, 'ppentq': 0.56, 'atq': 0.56, 'lctq': 0.56, 'dlcq': 0.56, 'apq': 0.559, 'dlttq': 0.445, 'ltq': 0.56, 'seqq': 0.56, 'xoprq': 0.55, 'cogsq': 0.563, 'xsgaq': 0.55}


In [36]:
# 3

def compute_fvol(df, fvol_cols=FVOL_COLS):
    d = df.sort_values(['code','date']).reset_index(drop=True).copy()
    pq = pd.PeriodIndex(d['date'], freq='Q')
    d['qidx'] = pq.year * 4 + (pq.quarter - 1)
    g = d.groupby('code', sort=False)
    has4 = (d['qidx'] - g['qidx'].shift(4)) == 4
    for c in fvol_cols:
        d[f'd_{c}'] = np.where(has4, d[c] - g[c].shift(4), np.nan)
    g = d.groupby('code', sort=False)
    def roll_std(s): return s.rolling(8, min_periods=4).std()
    def roll_mean(s): return s.rolling(8, min_periods=4).mean()
    for c in fvol_cols:
        d[f'std_{c}'] = g[f'd_{c}'].apply(roll_std).reset_index(level=0, drop=True)
    d['saleq_pos'] = d['saleq'].where(d['saleq'] > 0)
    g = d.groupby('code', sort=False)
    d['avg_sales'] = g['saleq_pos'].apply(roll_mean).reset_index(level=0, drop=True)
    d['avg_sales_valid'] = d['avg_sales'].where(d['avg_sales'] > 0)
    ind = []
    for c in fvol_cols:
        d[f'fvol_{c}'] = d[f'std_{c}'] / d['avg_sales_valid']
        ind.append(f'fvol_{c}')
    return d, ind

def finalize_fvol(d, ind, fvol_cols=FVOL_COLS):
    d = d.copy()
    rcols = [f'rank_{c}' for c in fvol_cols]
    ranked = d.groupby('date')[ind].rank(pct=True)
    ranked.columns = rcols
    d = pd.concat([d, ranked], axis=1)
    d['n_valid_rank'] = d[rcols].notna().sum(axis=1)
    d['FVOL'] = d[rcols].mean(axis=1)
    d.loc[d['n_valid_rank'] < 10, 'FVOL'] = np.nan
    return d

comp_fvol, ind = compute_fvol(comp_raw)
comp_fvol = finalize_fvol(comp_fvol, ind)
fvol = comp_fvol[comp_fvol['FVOL'].notna()][['code','date','FVOL']].copy()

print('comp_fvol:', comp_fvol.shape)
print('valid FVOL firm-quarters:', len(fvol))
print('avg firms per quarter:', round(fvol.groupby('date').size().mean(), 1))
print('range:', fvol['date'].min().date(), '~', fvol['date'].max().date())
print(fvol['FVOL'].describe().round(3).to_dict())

comp_fvol: (26166, 79)
valid FVOL firm-quarters: 13401
avg firms per quarter: 134.0
range: 2001-09-30 ~ 2026-06-30
{'count': 13401.0, 'mean': 0.504, 'std': 0.191, 'min': 0.03, '25%': 0.361, '50%': 0.492, '75%': 0.629, 'max': 1.0}


In [37]:
# 4

print("Table 1: Sample Descriptive Statistics for FVOL\n")

ind = [f'fvol_{c}' for c in FVOL_COLS]
comp_fvol['FVOL100'] = comp_fvol['FVOL'] * 100

def avg_dist(g, col):
    stats = pd.DataFrame({
        'Mean':  g[col].mean(),
        'Stdev': g[col].std(),
        'N':     g[col].count(),
        'p25':   g[col].quantile(.25),
        'p50':   g[col].quantile(.50),
        'p75':   g[col].quantile(.75),
    })
    return stats[stats['N'] > 0].mean()

g = comp_fvol.groupby('date')
panelA = pd.DataFrame({c: avg_dist(g, f'fvol_{c}') for c in FVOL_COLS}).T
panelA.loc['FVOL'] = avg_dist(g, 'FVOL100')
panelA['N'] = panelA['N'].round(0)

print('Panel A. Average Distribution of FVOL Measures')
print(panelA[['Mean','Stdev','N','p25','p50','p75']].round(2).to_string())

def avg_corr(df, cols, min_n=30):
    acc, k = None, 0
    for _, sub in df.groupby('date'):
        s = sub[cols].dropna()
        if len(s) < min_n:
            continue
        c = s.corr(method='spearman')
        acc = c if acc is None else acc.add(c)
        k += 1
    return acc / k

corr_cols = ind + ['FVOL']
labels = FVOL_COLS + ['FVOL']
pb = avg_corr(comp_fvol, corr_cols)
pb.index = labels
pb.columns = labels

print('\nPanel B. Average Spearman Correlations')
print(pb.round(2).to_string())

Table 1: Sample Descriptive Statistics for FVOL

Panel A. Average Distribution of FVOL Measures
         Mean  Stdev      N    p25    p50    p75
actq     0.44   1.01  134.0   0.14   0.23   0.40
rectq    0.16   0.34  134.0   0.05   0.09   0.16
invtq    0.09   0.11  128.0   0.03   0.06   0.11
ppentq   0.25   0.67  134.0   0.05   0.10   0.21
atq      0.69   1.76  134.0   0.17   0.31   0.58
lctq     0.44   1.11  134.0   0.14   0.23   0.41
dlcq     0.24   0.48  134.0   0.07   0.14   0.26
apq      0.17   0.36  134.0   0.06   0.10   0.16
dlttq    0.17   0.30  104.0   0.03   0.08   0.17
ltq      0.51   1.31  134.0   0.14   0.24   0.45
seqq     0.43   1.44  134.0   0.07   0.14   0.32
xoprq    0.18   0.40  135.0   0.06   0.10   0.17
cogsq    0.16   0.31  134.0   0.05   0.10   0.16
xsgaq    0.05   0.17  135.0   0.01   0.02   0.03
FVOL    50.88  19.08  134.0  36.84  49.73  62.93

Panel B. Average Spearman Correlations
        actq  rectq  invtq  ppentq   atq  lctq  dlcq   apq  dlttq   ltq  seqq  x

In [38]:
# 5

PRICE_ITEMS = {'S140500': 'reta', 'S102100': 'mktcap', 'S100300': 'prc'}
PRICE_PCT   = {'reta'}

def load_price_sheet(path, sheet='Prices'):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    icodes_all = raw.iloc[9, 1:]
    valid = icodes_all.isin(PRICE_ITEMS).to_numpy()
    codes  = raw.iloc[7, 1:].to_numpy()[valid]
    icodes = icodes_all.to_numpy()[valid]
    dates  = raw.iloc[14:, 0].to_numpy()
    vals   = raw.iloc[14:, 1:].apply(pd.to_numeric, errors='coerce').to_numpy()[:, valid]
    return codes, icodes, dates, vals

def price_to_tidy(codes, icodes, dates, vals, item_map=PRICE_ITEMS):
    n_item = len(item_map)
    n_col, n_date = len(codes), len(dates)
    assert n_col % n_item == 0, f'{n_col} not divisible by {n_item}'
    n_comp = n_col // n_item
    icode_blk = icodes.reshape(n_comp, n_item)
    assert (icode_blk == icode_blk[0]).all(), 'item order differs across companies'
    block_codes = list(icode_blk[0])
    assert set(block_codes) == set(item_map), f'item set mismatch: {block_codes}'
    comp_code = codes.reshape(n_comp, n_item)[:, 0]
    cube = vals.reshape(n_date, n_comp, n_item)
    out_cols = {item_map[ic]: cube[:, :, j].reshape(-1) for j, ic in enumerate(block_codes)}
    out = pd.DataFrame(out_cols)
    out.insert(0, 'date', np.repeat(pd.to_datetime(dates), n_comp))
    out.insert(0, 'code', np.tile(comp_code, n_date))
    out['ym'] = out['date'].dt.to_period('M')
    for c in PRICE_PCT:
        out[c] = out[c] / 100.0
    out = out[['code','ym'] + list(item_map.values())]
    return out.sort_values(['code','ym']).reset_index(drop=True)

def price_excel_to_parquet(path, sheet='Prices', out_path='Prices.parquet'):
    out_path = Path(out_path)
    if out_path.exists():
        print(f'{out_path.name} exists, skip')
        return pd.read_parquet(out_path)
    tidy = price_to_tidy(*load_price_sheet(path, sheet))
    tidy.to_parquet(out_path)
    print(f'{path} -> {out_path.name}  shape {tidy.shape}  '
          f'n_comp {tidy["code"].nunique()}  n_ym {tidy["ym"].nunique()}')
    return tidy

prices = price_excel_to_parquet('k200data.xlsx', sheet='Prices', out_path='Prices.parquet')

Prices.parquet exists, skip


In [39]:
# 7

print("Table 2: Low-High Returns\n")

fv = comp_fvol[comp_fvol['FVOL'].notna()][['code','date','FVOL']].copy()
fv['form_ym'] = fv['date'].dt.to_period('M') + 3

px = pd.read_parquet('Prices.parquet')

form = fv.merge(px[['code','ym','prc']].rename(columns={'ym':'form_ym'}),
                on=['code','form_ym'], how='left')
form = form[form['prc'] >= 5000].copy()

form['decile'] = (form.groupby('form_ym')['FVOL']
                  .transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop') + 1))
form = form.dropna(subset=['decile'])
form['decile'] = form['decile'].astype(int)

print('formation obs:', len(form))
print('avg firms/formation:', round(form.groupby('form_ym').size().mean(), 1))
print('formation range:', form['form_ym'].min(), '~', form['form_ym'].max())

holds = []
for k in [1, 2, 3]:
    h = form[['code','form_ym','decile']].copy()
    h['hold_ym'] = h['form_ym'] + k
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

rp = px[['code','ym','reta','mktcap']].copy().sort_values(['code','ym'])
rp['w'] = rp.groupby('code')['mktcap'].shift(1)

m = rp.merge(hold, left_on=['code','ym'], right_on=['code','hold_ym'], how='inner')
m = m.dropna(subset=['reta','w']).drop_duplicates(['code','form_ym','ym'])

dec_ret = (m.groupby(['ym','decile'])
           .apply(lambda g: (g['reta'] * g['w']).sum() / g['w'].sum(), include_groups=False)
           .reset_index(name='vwret'))

wide = dec_ret.pivot(index='ym', columns='decile', values='vwret')
lh = (wide[1] - wide[10]).dropna()

mean_ret = lh.mean()
t_stat = lh.mean() / (lh.std() / np.sqrt(len(lh)))

print('\nLow-High (decile 1 - 10), value-weighted, raw return')
print('n months :', len(lh))
print('mean     : {:.4f} ({:.2%}/month)'.format(mean_ret, mean_ret))
print('t-stat   : {:.2f}'.format(t_stat))
print('Low(1)   : {:.2%}   High(10): {:.2%}'.format(wide[1].mean(), wide[10].mean()))
print('\ndecile means (%/month):')
print((wide[list(range(1,11))].mean() * 100).round(3).to_string())

Table 2: Low-High Returns

formation obs: 11933
avg firms/formation: 121.8
formation range: 2002-03 ~ 2026-06

Low-High (decile 1 - 10), value-weighted, raw return
n months : 291
mean     : -0.0022 (-0.22%/month)
t-stat   : -2.31
Low(1)   : 0.02%   High(10): 0.23%

decile means (%/month):
decile
1     0.016
2     0.035
3     0.103
4     0.063
5     0.156
6     0.134
7     0.113
8     0.013
9     0.123
10    0.235


In [40]:
# 8

print("Table 2: Performance of FVOL Portfolios over Long Horizon\n")

fv = comp_fvol[comp_fvol['FVOL'].notna()][['code','date','FVOL']].copy()
fv['form_ym'] = fv['date'].dt.to_period('M') + 3

px = pd.read_parquet('Prices.parquet')

form = fv.merge(px[['code','ym','prc']].rename(columns={'ym':'form_ym'}),
                on=['code','form_ym'], how='left')
form = form[form['prc'] >= 5000].copy()
form['decile'] = (form.groupby('form_ym')['FVOL']
                  .transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop') + 1))
form = form.dropna(subset=['decile'])
form['decile'] = form['decile'].astype(int)

holds = []
for k in range(1, 25):
    h = form[['code','form_ym','decile']].copy()
    h['hold_ym'] = h['form_ym'] + k
    h['qtr'] = (k - 1) // 3 + 1
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

rp = px[['code','ym','reta','mktcap']].copy().sort_values(['code','ym'])
rp['w'] = rp.groupby('code')['mktcap'].shift(1)

m = rp.merge(hold, left_on=['code','ym'], right_on=['code','hold_ym'], how='inner')
m = m.dropna(subset=['reta','w']).drop_duplicates(['code','form_ym','ym','qtr'])

dr = (m.groupby(['qtr','ym','decile'])
      .apply(lambda g: (g['reta'] * g['w']).sum() / g['w'].sum(), include_groups=False)
      .reset_index(name='r'))

def lh_stat(s):
    return s.mean() * 100, s.mean() / (s.std() / np.sqrt(len(s))), len(s)

print(f"{'Q':5}{'Ret%':>9}{'t':>8}{'n':>6}")
parts = []
for q in range(1, 9):
    w = dr[dr['qtr'] == q].pivot(index='ym', columns='decile', values='r')
    lh = (w[1] - w[10]).dropna()
    parts.append(lh.rename(f'q{q}'))
    ret, t, n = lh_stat(lh)
    print(f"Q{q:<4}{ret:9.3f}{t:8.2f}{n:6d}")

jt = pd.concat(parts, axis=1).mean(axis=1).dropna()
ret, t, n = lh_stat(jt)
print(f"{'JT8':5}{ret:9.3f}{t:8.2f}{n:6d}")
print('\n(Original(Paper): Q1~Q8: +0.93,+0.96,+0.88,+0.83,+0.77,+0.65,+0.75,+0.70 / JT8 +0.81, ALL positive)')

Table 2: Performance of FVOL Portfolios over Long Horizon

Q         Ret%       t     n
Q1      -0.218   -2.31   291
Q2      -0.254   -2.56   288
Q3      -0.291   -2.91   285
Q4      -0.213   -2.14   282
Q5      -0.154   -1.66   279
Q6      -0.192   -2.01   276
Q7      -0.171   -1.68   273
Q8      -0.161   -1.58   270
JT8     -0.204   -2.31   291

(Original(Paper): Q1~Q8: +0.93,+0.96,+0.88,+0.83,+0.77,+0.65,+0.75,+0.70 / JT8 +0.81, ALL positive)


In [41]:
# 9

fin = pd.read_parquet('Financials.parquet')

RENAME_PROF = {
    'M121000.M': 'revtq', 'M121100.M': 'cogsq', 'M121300.M': 'xsgaq',
    'M121900.M': 'xintq', 'M122700.M': 'ibq', 'M111000.M': 'atq',
    'M115000.M': 'seqq', 'M114300.M': 'txditcq', 'M115200.M': 'pstkq',
}
prof = fin.rename(columns=RENAME_PROF)[['code','date','revtq','cogsq','xsgaq','xintq','ibq','atq','seqq','txditcq','pstkq']].copy()
prof = prof.sort_values(['code','date']).reset_index(drop=True)

g = prof.groupby('code', sort=False)
prof['qbe'] = prof['seqq'] + prof['txditcq'].fillna(0) - prof['pstkq'].fillna(0)
prof['atq_lag'] = g['atq'].shift(1)
prof['qbe_lag'] = g['qbe'].shift(1)

prof['gp']  = (prof['revtq'] - prof['cogsq']) / prof['atq_lag'].where(prof['atq_lag'] > 0)
prof['op']  = (prof['revtq'] - prof['cogsq'] - prof['xsgaq'].fillna(0) - prof['xintq'].fillna(0)) / prof['qbe_lag'].where(prof['qbe_lag'] > 0)
prof['roe'] = prof['ibq'] / prof['qbe_lag'].where(prof['qbe_lag'] > 0)

prof_calc = prof[['code','date','gp','op','roe']].copy()

print('prof_calc:', prof_calc.shape)
print('notna:', prof_calc[['gp','op','roe']].notna().mean().round(3).to_dict())
print(prof_calc[['gp','op','roe']].describe(percentiles=[.05,.5,.95]).round(3).to_string())

prof_calc: (26166, 5)
notna: {'gp': 0.543, 'op': 0.541, 'roe': 0.541}
              gp         op        roe
count  14196.000  14145.000  14149.000
mean       0.058      0.026      0.019
std        0.066      0.083      0.206
min       -1.263     -4.858    -16.130
5%         0.004     -0.031     -0.042
50%        0.043      0.023      0.021
95%        0.167      0.097      0.089
max        2.022      1.290      2.907


In [42]:
# 10

print("Figure 3: Profitability of FVOL Decile Portfolios\n")

fv = comp_fvol[comp_fvol['FVOL'].notna()][['code','date','FVOL']].copy()
fv['form_ym'] = fv['date'].dt.to_period('M') + 3
fv['prof_q'] = fv['date'].dt.to_period('Q')

px = pd.read_parquet('Prices.parquet')
form = fv.merge(px[['code','ym','prc']].rename(columns={'ym':'form_ym'}),
                on=['code','form_ym'], how='left')
form = form[form['prc'] >= 5000].copy()
form['decile'] = (form.groupby('form_ym')['FVOL']
                  .transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop') + 1))
form = form.dropna(subset=['decile'])
form['decile'] = form['decile'].astype(int)

pc = prof_calc.copy()
pc['prof_q'] = pc['date'].dt.to_period('Q')
d = form.merge(pc[['code','prof_q','gp','op','roe']], on=['code','prof_q'], how='left')

def winz(s):
    return s.clip(s.quantile(.01), s.quantile(.99))
for c in ['gp', 'op', 'roe']:
    d[c] = d.groupby('form_ym')[c].transform(winz)

t = d.groupby('decile')[['gp','op','roe']].mean() * 100
print(f"{'dec':4}{'GP%':>9}{'OP%':>9}{'ROE%':>9}")
for dd in range(1, 11):
    print(f"{dd:<4}{t.loc[dd,'gp']:9.2f}{t.loc[dd,'op']:9.2f}{t.loc[dd,'roe']:9.2f}")
print("\n(paper: GP 14->4, OP 8->-0.1, ROE 4->-3 ; monotone decreasing in FVOL decile)")

Figure 3: Profitability of FVOL Decile Portfolios

dec       GP%      OP%     ROE%
1        7.48     3.38     2.81
2        6.68     3.08     2.64
3        6.66     2.91     2.45
4        5.93     2.94     2.37
5        5.69     2.93     2.55
6        5.14     2.59     2.12
7        5.13     2.92     2.33
8        5.15     2.55     2.48
9        4.25     1.78     1.28
10       4.08     1.11     1.02

(paper: GP 14->4, OP 8->-0.1, ROE 4->-3 ; monotone decreasing in FVOL decile)


In [43]:
# 11

print("Figure 2: Market Cap Distribution of FVOL Deciles\n")

fv = comp_fvol[comp_fvol['FVOL'].notna()][['code','date','FVOL']].copy()
fv['form_ym'] = fv['date'].dt.to_period('M') + 3

px = pd.read_parquet('Prices.parquet')
form = fv.merge(px[['code','ym','prc','mktcap']].rename(columns={'ym':'form_ym'}),
                on=['code','form_ym'], how='left')
form = form[form['prc'] >= 5000].copy()
form['decile'] = (form.groupby('form_ym')['FVOL']
                  .transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop') + 1))
form = form.dropna(subset=['decile', 'mktcap'])
form['decile'] = form['decile'].astype(int)

form['tot'] = form.groupby('form_ym')['mktcap'].transform('sum')
form['share'] = form['mktcap'] / form['tot']
dec_share = form.groupby(['form_ym','decile'])['share'].sum().reset_index()
avg_share = dec_share.groupby('decile')['share'].mean() * 100

print(f"{'dec':4}{'size share %':>14}")
for dd in range(1, 11):
    print(f"{dd:<4}{avg_share.loc[dd]:14.2f}")
print("\n(paper: decile 1 = 16.3%, decile 10 = 5.2% ; low FVOL = larger)")

Figure 2: Market Cap Distribution of FVOL Deciles

dec   size share %
1            16.28
2            17.39
3            11.47
4            10.34
5             7.98
6             7.11
7             7.28
8             7.93
9             8.01
10            6.21

(paper: decile 1 = 16.3%, decile 10 = 5.2% ; low FVOL = larger)


# 진행 로그

## 표본
- 현 KOSPI200 − 금융주 22개 = 178개. 생존편향 있음(현시점 구성종목 고정), 코드 검증용. 본판은 코스피 전체 1118종목으로 재실행 예정.
- 제거한 금융주 22개:
  - A000810 삼성화재, A001450 현대해상, A005830 DB손해보험, A005940 NH투자증권, A006800 미래에셋증권, A016360 삼성증권, A024110 기업은행, A029780 삼성카드, A032830 삼성생명, A039490 키움증권, A055550 신한지주, A071050 한국금융지주, A086790 하나금융지주, A088350 한화생명, A105560 KB금융, A138040 메리츠금융지주, A138930 BNK금융지주, A139130 iM금융지주, A175330 JB금융지주, A316140 우리금융지주, A323410 카카오뱅크, A377300 카카오페이

## 데이터 (퀀티와이즈 --> parquet 캐시)
- Financials.parquet : 재무 19항목, 분기, Financials 시트 → 26,166행(178×147), 1989-12~2026-06
- Prices.parquet : 수정주가수익률(현금배당포함, S140500)·시가총액(S102100)·수정주가(S100300), 월, Prices 시트 → 78,142행(178×439)
- DailyReturns.parquet : 수정주가수익률(현금배당포함, S140500), 일별, DailyReturns 시트 → 1,044,237행(178종목), 1990-01~2026-06. 상장 전 NaN은 dropna.
- JKP Global Factor Data : 한국, 월별, capped value weighted, USD. 13개 테마 중 size·value·profitability·investment·momentum 5개 확보. market·q-factor 없음. 통화(USD↔원화) 미정렬 상태 → 본판에서 처리.
- 엑셀 레이아웃: 7=Code, 9=Item Code, 13=한글명, 14행~데이터. 회사당 N칸 블록 반복. Item Code 마스크로 꼬리 빈 열 제거.
- reta는 % → 소수(÷100). 날짜는 월말; merge 키는 to_period('M').

## 재무항목 매핑 (FnGuide → Compustat)
FVOL 14개:
- 유동자산→actq, 매출채권및기타채권→rectq, 재고자산→invtq, 유형자산→ppentq, 자산총계→atq
- 유동부채→lctq, 만기도래차입금→dlcq, 매입채무및기타채무→apq, 장기차입금→dlttq, 부채총계→ltq
- 자본총계→seqq, 매출원가→cogsq, 판관비→xsgaq, 영업비용→xoprq(=매출원가+판관비, 파생)
분모/수익성:
- 매출액→saleq, 영업이익, 이자비용→xintq, 당기순이익→ibq, 이연법인세부채→txditcq, 우선주자본금→pstkq

매핑 메모(한국 회계 정합):
- dlcq: 한국에 '단기차입금' 별도 계정 없음 → 만기도래차입금(M113300) 사용.
- seqq: 자본총계 = 총자본(비지배 포함, M115000). 미국 seqq는 지배지분이나 FVOL은 변동성만 쓰므로 총자본 그대로 사용.
- ibq: K-IFRS 특별손익 폐지 → 당기순이익 ≈ income before extraordinary items. 그대로 사용.
- xoprq: 미추출 → 매출원가+판관비로 파생.
- txditcq: 한국은 투자세액공제 미계상, 이연법인세부채 사용.

## FVOL 측정
- 4분기 차분 → 8분기 롤링 std(min 4) ÷ 평균매출(8분기) → 분기별 percentile rank → 14개 평균 = FVOL
- rank는 178종목 내에서 분기별로 매김. n_valid≥10 임계.
- 유효 FVOL: 13,401 firm-q, 분기당 평균 134개, 2001-09~2026-06. FVOL은 0–1 스케일(rank 평균).

## 포트폴리오 형성 규칙 (Table 2~)
- 공시 시차: rdq 없음 → 분기말 +1분기(+3개월) 고정 lag.
- 가격 필터: 형성 시점 주가 ≥ 5,000원 (저가주 12.5% 제거).
- decile 10분위, Low(1)−High(10), 3개월 보유, 시총가중(직전월 시총).
- 상장폐지 보정: KOSPI200 현 종목이라 현 단계 무관, 본판에서 처리.

## 복원 현황
### Table 1 (분포 / Spearman 상관): 재현 성공
- FVOL mean 50.88, p50 49.73, std 19.08 (논문 49.48/48.57/20.19과 근접).
- FVOL이 14개 측정치와 전부 +상관 0.39~0.81 (논문 0.54~0.87 대역). 공통성분 확인.
- 논문 Table 1은 산업조정 기준 / 현재는 raw. 산업조정판 추후.

### Table 2 / Table 3 (Low-High raw return): 코드 작동 확인, 결과는 보류
- Q1 Low−High VW raw return −0.22%/월 (t=−2.31). 미국(+0.93%)과 부호 반대.
- Table 3 Q1~Q8 + JT8 전부 음수, 8분기 일관. JT8 −0.20%(t=−2.31).
- 진단: lag·decile 정렬·가중 모두 정상(코드 버그 아님).
  - VW −0.22%(t=−2.31) vs EW −0.095%(t=−1.40): 효과가 시총 큰 종목에 의존.
  - 음수가 전 기간 일관(2002-09 −0.33%, 2010-17 −0.23%, 2018-26 −0.11%).
  - 형성당 종목 52~174(중앙 125), 초기 얇음.

### Figure 2 (시총 분포): 논문과 거의 일치
- decile 1(low FVOL) 16.3%, decile 10(high) 6.2%. 논문 16.3%/5.2%와 포개짐.
- "저FVOL=대형, 고FVOL=소형" 구조 한국에서도 성립.

### Figure 3 (수익성 by decile): 메커니즘 성립
- decile 1→10으로 GP 7.5→4.1%, OP 3.4→1.1%, ROE 2.8→1.0%. 세 지표 단조 감소.
- "고FVOL → 저수익성" 핵심 메커니즘 한국에서도 성립(미국보다 기울기 완만).

### 종합
- 측정(Table 1)·구조(Figure 2)·수익성 메커니즘(Figure 3)은 모두 논문과 일치.
- 수익률(Table 2·3)만 부호 반대. 인과사슬 "고FVOL→저수익성→저수익률"에서 앞 고리는 성립, 뒤 고리만 끊김.
- 해석: 생존편향(현 KOSPI200 고정)이 수익률만 왜곡한 정황 강함 — 변동성 크고 수익성 낮았으나 살아남아 대형주가 된 종목들이 사후 고수익률을 주도. 코드 검증은 성공, 결과 판단은 본판(코스피 전체)으로.

## 남은 작업
- 팩터: 직접 구성 결정. 코스피 전체 유니버스 필요 → 본판에서. (JKP는 통화·market 보완 후 보조 활용 가능)
- 산업분류(GICS): 산업조정 FVOL용, 미추출.
- 알파·IVOL·spanning: 팩터 확보 후.

## 미국 원본과 방법론 차이
이미 명시된 것:
- 표본: 현 KOSPI200 − 금융주 22개 (원본: CRSP 전체 보통주, 시점별 동적).
- 재무항목 매핑: FnGuide → Compustat (dlcq=만기도래차입금, seqq=총자본, ibq=당기순이익, xoprq=cogs+xsga 파생 등).
- 공시 시차: rdq 없음 → 분기말 +1분기 고정 lag (원본: 실제 rdq).
- 가격 필터: 5,000원 고정 (원본: $5).

추가로 명시할 차이:
- 상장폐지 보정 생략: 원본은 Shumway(1997) delisting return 보정(−30% 등). 현 KOSPI200이라 생략 → 본판에서 필수.
- percentile rank 유니버스: 178종목 내에서 rank (원본: 매 분기 500+ 전체 종목). 작은 풀에서의 상대순위라 의미가 다름.
- 수익성 OP 정의: revt − cogs − xsga − xint, QBE로 나눔. 원본과 동일하게 구현했으나 음수/예외 처리 디테일은 완전 일치 미확인.
- JT8 구현: +1~24개월 보유분 겹쳐 평균. 원본 Jegadeesh-Titman 비중복 구성과 미세하게 다를 수 있음.
- winsorize: 수익성에만 1/99% 적용. 수익률(reta, dret)은 미적용 → 극단치 처리 필요(dret min −77%, 월 reta ±30% 캡 의심).

수치 비교 주의:
- 표본기간 2001~2026 vs 1976~2024, 분기 종목수 ~134 vs ~1,900. 1:1 비교 무의미.

## 본질적 쟁점
- FF5의 한국 적용: 선행연구상 한국에서 FF5 작동방식이 미국과 다름(예: CMA 부호 반대). 알파 해석의 근본 쟁점이며 이 연구의 핵심.

## 평가
- 재현 실패 지점은 코드가 아니라 전부 표본/데이터 특성. 측정·구조·수익성 메커니즘은 이식 성공.
- 원 코드가 표본 독립적으로 작성돼(키 기반 groupby, 동적 rank) 한국 데이터로 키만 교체해 재사용 가능.